# 06: Real-World Holdout Validation, Continuous ANOVA Gating & Agentune AutoML Overfitting Guardrails

### Empirical Evidence Across 3 Production Datasets, 4 Frontier LLMs & Automated Feature Synthesis

---

### Executive Overview & Motivation

In earlier notebooks, we demonstrated that unassisted LLM exploratory agents suffer from **Search Space Amnesia** and **Batch Blindness** on synthetic control datasets (`null_honeypot`, `needle_in_haystack`, `correlated_blocks`).

To address whether those results were specific to synthetic data, this notebook presents three production-grade empirical extensions:

1. **Part 1: Real-World Production Datasets Evaluated Across 4 Frontier LLMs**
   - **Lending Club Loan Defaults** ($N = 54,655$ loans, 42 columns, target: `bad_loans`).
   - **King County House Sales** ($N = 21,610$ transactions, target: `high_value` & continuous `price`).
   - **UCI Bank Marketing Campaigns** ($N = 41,188$ client calls, low-SNR 11.3% subscription base rate, target: `y`).
   - True **Out-of-Sample (OOS) Holdout Partitioning**: Agents explore strictly on $N_{\text{train}} = 5,000$, and promoted drivers are audited against the **entire untouched holdout test partition** ($N_{\text{test}} = 10,985$ on Lending Club, $4,322$ on King County, $8,202$ on Bank Marketing).
   - Live evaluations across **4 Frontier Models**:
     - **Anthropic Claude Opus 5.5** (`claude-opus-5-5`)
     - **Anthropic Claude Sonnet 5.5** (`claude-sonnet-5-5`)
     - **OpenAI GPT-6.1-Sol** (`gpt-6.1-sol`)
     - **Google Gemini 3.5 Flash Lite** (`gemini-3.5-flash-lite`)

2. **Part 2: Closed-Form Continuous Metric Gating (`ANOVA F -> RIG`)**
   - Using the Gaussian / ANOVA mutual information identity:
     $$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}, \qquad I(X;Y) = -\frac{1}{2}\log_2(1 - \eta^2), \qquad G_{\text{equiv}} = -N \ln(1 - \eta^2)$$
   - SignalGate gates continuous SQL `GROUP BY` aggregate queries returning `COUNT(*)`, `AVG(y)`, and `STDDEV(y)` directly from summary statistics without arbitrary target binarization.

3. **Part 3: SparkBeyond Agentune AutoML Overfitting Benchmark**
   - Automated feature engineering engines (e.g., [SparkBeyond Agentune](https://github.com/SparkBeyond/agentune)) synthesize $M = 1,000+$ candidate interaction features ($A \land B$, $A \lor B$, subgroup slices).
   - We benchmark **Unassisted Agentune (Greedy AutoML)** vs. **SignalGate-Governed Selection** on holdout replication rate, out-of-sample decay, and downstream classifier generalization (ROC AUC).


---
## Part 1: Real-World Empirical Holdout Audit Across 4 Frontier LLMs

We benchmark three agent archetypes (`business_analyst`, `statistician_unassisted`, `statistician_signalgate`) across 3 real-world datasets and 4 frontier models (36persona-model-dataset configurations $\times$ 3 seeds = **108 live frontier evaluations**).

When unassisted agents drill into realistic small subgroup slices ($35 \le N_{\text{slice}} \le 165$), finite-sample contingency noise inflates observed training RIG up to $0.18 - 0.30$. SignalGate's stateful `NoiseGateSessionLedger` enforces both **retroactive terminal session multiple-testing thresholds** $\tau_{\text{sig}}(M_{\text{eff}})$ and **Cochran's small-cell sparsity guard** ($E_{rc} < 5$).


In [1]:
import pandas as pd
from IPython.display import display

# Load master summary across all 3 real-world datasets and 4 frontier models
all_models_summary = pd.read_csv('../Data/real_world_eval_all_models_summary.csv')
print(f"Total Evaluated Configurations: {len(all_models_summary)} rows (108 live frontier runs)")
display(all_models_summary)


Total Evaluated Configurations: 36 rows (108 live frontier runs)
       Dataset                 Model                 Persona  Holdout_Replication_Rate (%)  Real_FDR (%)  Mean_Train_RIG  Mean_Holdout_RIG  Generalization_Bias  Decay (%)  Under_Reporting_Ratio  Completed_Seeds
bank_marketing       claude-opus-5-5        business_analyst                    100.000000      0.000000        0.056978          0.052589             0.004389  10.536667               0.277333                3
bank_marketing       claude-opus-5-5 statistician_signalgate                    100.000000      0.000000        0.056978          0.052589             0.004389  10.536667               0.986111                3
bank_marketing       claude-opus-5-5 statistician_unassisted                    100.000000      0.000000        0.056978          0.052589             0.004389  10.536667               0.249451                3
bank_marketing     claude-sonnet-5-5        business_analyst                    100.000000 

### Figure 4: Out-of-Sample Replication Rate & Real False Discovery Rate Across Frontier Models


In [1]:
from IPython.display import Image, display
display(Image(filename='figures/fig4_real_world_frontier_replication.png'))


### Key Findings Across Frontier Models & Real-World Datasets:

1. **100.0% Out-of-Sample Replication Across All 4 Frontier Models with SignalGate**:
   - Across `claude-opus-5-5`, `claude-sonnet-5-5`, `gpt-6.1-sol`, and `gemini-3.5-flash-lite`, the `statistician_signalgate` persona achieved **100.0% Holdout Replication Rate (0.0% Real FDR)** on all three production datasets (`lending_club`, `king_county`, and `bank_marketing`).
2. **Unassisted Business Analysts Suffer Severe Out-of-Sample Collapse**:
   - On Lending Club, unassisted `business_analyst` agents promoted small-slice traps (`[purpose==small_business]`, `[home_ownership==OTHER]`) that appeared to have high training RIG ($0.10 - 0.24$) but collapsed on holdout:
     - `gpt-6.1-sol`: **100.0% Real FDR** (0% holdout replication)
     - `claude-sonnet-5-5`: **77.8% Real FDR** (22.2% holdout replication)
     - `gemini-3.5-flash-lite`: **40.0% Real FDR** (60.0% holdout replication)
     - `claude-opus-5-5`: **19.4% Real FDR** (80.6% holdout replication, 30.6% holdout decay)
3. **Why SignalGate Beats the Unassisted Statistician on King County**:
   - On King County housing, `statistician_signalgate` (`claude-opus-5-5`, `claude-sonnet-5-5`, `gpt-6.1-sol`) achieved **higher Holdout RIG ($0.2184$ vs $0.1676$)** and **lower Holdout Decay ($2.64\%$ vs $3.36\%$)** than `statistician_unassisted`, while maintaining **$1.00\times$ exact hypothesis ledger tracking**.
   - Meanwhile, under Batch Blindness, unassisted models either severely under-reported the search space ($4.8\times - 7.6\times$ on `gpt-6.1-sol` and `gemini-3.5-flash-lite`) or wildly over-guessed hidden grid sizes ($0.18\times - 0.31\times$ on `claude-opus-5-5`).


---
## Part 2: Closed-Form Continuous Metric Gate (`ANOVA F -> RIG`)

Real-world enterprise tables frequently store continuous metrics (e.g., `price` in King County Housing, revenue, latency). Rather than forcing arbitrary target binarization, SignalGate directly gates SQL `GROUP BY` aggregate outputs (`COUNT(*)`, `AVG(y)`, `STDDEV(y)`) via one-way ANOVA variance decomposition:

$$SS_{\text{between}} = \sum_{k=1}^{K} n_k (\bar{y}_k - \bar{y})^2, \qquad SS_{\text{within}} = \sum_{k=1}^{K} (n_k - 1) s_k^2, \qquad \eta^2 = \frac{SS_{\text{between}}}{SS_{\text{between}} + SS_{\text{within}}}$$

Under the Gaussian mutual information equivalence, Wilks' likelihood ratio statistic maps directly to SignalGate's $\chi^2_{K-1}$ Extreme Value Theory envelope:
$$G_{\text{equiv}} = -N \ln(1 - \eta^2) \xrightarrow{d} \chi^2_{K-1} \text{ under } H_0$$


In [1]:
from signalgate import anova_rig_from_group_stats, NoiseGateSessionLedger

# Example: Gating a SQL GROUP BY query on continuous King County Home Prices
# SELECT grade_tier, COUNT(*), AVG(log_price), STDDEV(log_price) FROM king_county GROUP BY grade_tier
ledger = NoiseGateSessionLedger()

res_genuine = ledger.evaluate_continuous_group_aggregation(
    session_id="kc_continuous_sql",
    feature_name="grade_tier",
    group_counts=[1450, 2600, 950],
    group_means=[12.72, 13.08, 13.68],
    group_stds=[0.36, 0.39, 0.44],
    hypotheses_in_batch=25,
)

# Spurious small-slice SQL GROUP BY with tiny cell count (n_k = 3 < 5)
res_spurious_slice = ledger.evaluate_continuous_group_aggregation(
    session_id="kc_continuous_sql",
    feature_name="[waterfront==1 AND bedrooms==6] view_rating",
    group_counts=[3, 4, 2],
    group_means=[13.10, 14.85, 13.40],
    group_stds=[0.40, 0.25, 0.50],
    hypotheses_in_batch=25,
)

print("1. Genuine Continuous Driver (grade_tier):")
print(f"   eta^2 = {res_genuine['anova_metrics']['eta_squared']:.4f} | F = {res_genuine['anova_metrics']['f_stat']:.1f} | "
      f"Equiv RIG = {res_genuine['observed_rig']:.4f} | Gate = {res_genuine['bonferroni_gate_rig']:.4f} -> {res_genuine['verdict']}")

print("\n2. Sparse Small-Slice Continuous Query ([waterfront==1 AND bedrooms==6] view_rating):")
print(f"   eta^2 = {res_spurious_slice['anova_metrics']['eta_squared']:.4f} | min_n_k = {res_spurious_slice['expected_co_occurrences_E11']:.0f} | "
      f"Small-Cell Warning = {res_spurious_slice['small_cell_warning']} -> {res_spurious_slice['verdict']}")


1. Genuine Continuous Driver (grade_tier):
   eta^2 = 0.3712 | F = 1475.1 | Equiv RIG = 0.2896 | Gate = 0.0029 -> VERIFIED_SIGNAL

2. Sparse Small-Slice Continuous Query ([waterfront==1 AND bedrooms==6] view_rating):
   eta^2 = 0.8759 | min_n_k = 2 | Small-Cell Warning = True -> BORDERLINE_NEEDS_VALIDATION


---
## Part 3: SparkBeyond Agentune AutoML Overfitting Benchmark

Tools like [SparkBeyond Agentune](https://github.com/SparkBeyond/agentune) synthesize thousands of candidate interaction features ($A \land B$, $A \lor B$, subgroup slices) and rank them by information gain on training data. When evaluating $M = 1,000$ candidates on $N_{\text{train}} = 5,000$, greedy selection falls victim to the **Winner's Curse**.


In [1]:
# Load Agentune benchmark results on Lending Club
agentune_summary = pd.read_csv('../Data/agentune_benchmark_lending_club_summary.csv')
agentune_downstream = pd.read_csv('../Data/agentune_benchmark_lending_club_downstream.csv')

print("=== AGENTUNE AUTOML FEATURE SELECTION BENCHMARK (M = 1000 Candidates) ===")
display(agentune_summary)

print("\n=== DOWNSTREAM ML CLASSIFIER GENERALIZATION (Overfitting Gap) ===")
display(agentune_downstream)


=== AGENTUNE AUTOML FEATURE SELECTION BENCHMARK (M = 1000 Candidates) ===
                           Approach  Total_Generated_M  Selected_Features  Holdout_Replication_Rate (%)  Real_FDR (%)  Mean_Train_RIG  Mean_Holdout_RIG  Generalization_Bias  Decay (%)
Unassisted Agentune (Greedy AutoML)               1000                 15                     73.333333     26.666667        0.035556          0.022455             0.013101  39.195033
      SignalGate-Governed Selection               1000                 15                    100.000000      0.000000        0.031442          0.026031             0.005410  21.887352

=== DOWNSTREAM ML CLASSIFIER GENERALIZATION (Overfitting Gap) ===
                            Pipeline  N_Features  Train_AUC  Holdout_Test_AUC  Overfitting_Gap
               Baseline Raw Features         119   0.729152          0.700973         0.028179
Baseline + Unassisted Agentune Top-K         134   0.730266          0.699346         0.030920
Baseline + SignalGate-

### Figure 5: Feature Holdout Decay Curves (Unassisted Agentune vs. SignalGate)


In [1]:
display(Image(filename='figures/fig5_agentune_holdout_decay.png'))


### Figure 6: The Downstream AutoML Overfitting Gap


In [1]:
display(Image(filename='figures/fig6_agentune_downstream_overfitting_gap.png'))


---
## Quantitative Summary

| Benchmark Dimension | Unassisted Baseline / Agentune | SignalGate-Governed Selection | SignalGate Advantage |
| :--- | :---: | :---: | :---: |
| **Frontier LLM Real FDR (Lending Club)** | **19.4% – 100.0%** (`business_analyst`) | **0.0%** (All 4 Frontier LLMs) | **100% Holdout Replication** |
| **Frontier LLM Real FDR (King County)** | **0.0% – 66.7%** (`business_analyst`) | **0.0%** (All 4 Frontier LLMs) | **Higher Holdout RIG (0.218 vs 0.168)** |
| **Frontier LLM Real FDR (Bank Marketing)** | **0.0% – 41.7%** (`business_analyst`) | **0.0%** (All 4 Frontier LLMs) | **1.00x Exact Ledger Tracking** |
| **Agentune AutoML Replication ($M=1,000$)** | **73.3%** (26.7% Real FDR) | **100.0%** (0.0% Real FDR) | **Zero Spurious Interactions** |
| **Agentune Out-of-Sample Decay %** | **39.2%** | **21.9%** | **1.8x Lower Decay** |
| **Downstream Holdout ROC AUC** | **0.6993** (Degraded vs raw baseline) | **0.7029** (Beat raw baseline) | **Eliminates Interaction Overfitting** |
